# Criminal Data


### 1. Initial Setup


In [1]:
import pandas as pd

In [2]:
path = "../data/2026_raw.csv"

### 2. Raw Data


In [3]:
with open(path, "r", encoding="latin-1") as f:
    for _ in range(3):
        print(f.readline()[:120])

Sequência;Data Fato;Hora Fato;Grupo Fato;Tipo Enquadramento;Tipo Fato;Municipio Fato;Local Fato;Bairro;Quantidade Vítima
1;01/01/2026;00:01:00;CRIMES;FURTO ABIGEATO;Consumado;GENERAL CAMARA;OUTROS;;1;45;Masculino;Sem informação;;;;;;;;;;;;;;
2;01/01/2026;00:01:00;CONTRAVENCOES;PERTURBACAO DO TRABALHO OU DO SOSSEGO ALHEIOS;Consumado;IJUI;RESIDENCIA;Centro;1;62;


### 3. Sample Data


In [4]:
df = pd.read_csv(path, sep=";", encoding="latin-1", nrows=5)

In [5]:
df.head()

,Sequência,Data Fato,Hora Fato,Grupo Fato,Tipo Enquadramento,Tipo Fato,Municipio Fato,Local Fato,Bairro,Quantidade Vítimas,...,Unnamed: 102,Unnamed: 103,Unnamed: 104,Unnamed: 105,Unnamed: 106,Unnamed: 107,Unnamed: 108,Unnamed: 109,Unnamed: 110,Unnamed: 111
0,1,01/01/2026,00:01:00,CRIMES,FURTO ABIGEATO,Consumado,GENERAL CAMARA,OUTROS,NaN,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,01/01/2026,00:01:00,CONTRAVENCOES,PERTURBACAO DO TRABALHO OU DO SOSSEGO ALHEIOS,Consumado,IJUI,RESIDENCIA,Centro,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,01/01/2026,00:01:00,CRIMES,LESAO CORPORAL,Consumado,NOVO HAMBURGO,RESIDENCIA,Vila Nova,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,01/01/2026,00:01:00,CONTRAVENCOES,VIAS DE FATO,Consumado,PAROBE,RESIDENCIA,Vila Jardim,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,01/01/2026,00:01:00,CRIMES,PERSEGUICAO,Consumado,TAPEJARA,RESIDENCIA,Treze de Maio,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df.shape

(5, 112)

In [7]:
df.columns[9:20]

Index(['Quantidade Vítimas', 'Idade Vítima', 'Sexo Vítima', 'Cor Vítima',
       '...', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17',
       'Unnamed: 18', 'Unnamed: 19'],
      dtype='str')

### 4. Data Overview


In [8]:
df = pd.read_csv(path, sep=";", encoding="latin-1", usecols=range(13))

In [9]:
df.shape

(502414, 13)

### 5. Memory Usage


##### 5.1. Before Conversion


In [10]:
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 502414 entries, 0 to 502413
Data columns (total 13 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Sequência           502414 non-null  int64  
 1   Data Fato           502414 non-null  str    
 2   Hora Fato           502414 non-null  str    
 3   Grupo Fato          502414 non-null  str    
 4   Tipo Enquadramento  502414 non-null  str    
 5   Tipo Fato           502414 non-null  str    
 6   Municipio Fato      502414 non-null  str    
 7   Local Fato          502414 non-null  str    
 8   Bairro              404556 non-null  str    
 9   Quantidade Vítimas  502407 non-null  float64
 10  Idade Vítima        407370 non-null  float64
 11  Sexo Vítima         407148 non-null  str    
 12  Cor Vítima          407618 non-null  str    
dtypes: float64(2), int64(1), str(10)
memory usage: 95.7 MB


In [11]:
df.memory_usage(deep=True).sort_values(ascending=False)

Tipo Enquadramento    13796228
Municipio Fato         9389439
Data Fato              9043452
Local Fato             8755482
Tipo Fato              8541038
Bairro                 8276012
Hora Fato              8038624
Cor Vítima             7620357
Sexo Vítima            7527783
Grupo Fato             7251818
Idade Vítima           4019312
Sequência              4019312
Quantidade Vítimas     4019312
Index                      132
dtype: int64

### 6. Data Types


##### 6.1. Categorical Data


In [12]:
categories = [
    "Grupo Fato",
    "Tipo Enquadramento",
    "Tipo Fato",
    "Municipio Fato",
    "Local Fato",
    "Bairro",
    "Sexo Vítima",
    "Cor Vítima",
]

In [13]:
before = df[categories].memory_usage(deep=True).sum()
df[categories] = df[categories].astype("category")
after = df[categories].memory_usage(deep=True).sum()

In [14]:
print(f"Memory usage before: {before / 1e6:.1f} MB")
print(f"Memory usage after: {after / 1e6:.1f} MB")

Memory usage before: 71.2 MB
Memory usage after: 5.9 MB


##### 6.2. DateTime Data


In [15]:
df["Data Hora"] = pd.to_datetime(
    df["Data Fato"] + " " + df["Hora Fato"],
    format="%d/%m/%Y %H:%M:%S",
)
df = df.drop(columns=["Data Fato", "Hora Fato"])
df["Data Hora"].agg(["min", "max"])


min   2026-01-01 00:01:00
max   2026-08-31 23:50:42
Name: Data Hora, dtype: datetime64[us]

In [16]:
df["Data Hora"].dt.day_name().value_counts()

Data Hora
Friday       77646
Monday       77011
Thursday     76260
Wednesday    75947
Tuesday      73485
Saturday     63572
Sunday       58493
Name: count, dtype: int64

##### 6.3. Numeric Data


In [17]:
integers = ["Quantidade Vítimas", "Idade Vítima"]
df[integers] = df[integers].astype("Int8")
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 502414 entries, 0 to 502413
Data columns (total 12 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   Sequência           502414 non-null  int64         
 1   Grupo Fato          502414 non-null  category      
 2   Tipo Enquadramento  502414 non-null  category      
 3   Tipo Fato           502414 non-null  category      
 4   Municipio Fato      502414 non-null  category      
 5   Local Fato          502414 non-null  category      
 6   Bairro              404556 non-null  category      
 7   Quantidade Vítimas  502407 non-null  Int8          
 8   Idade Vítima        407370 non-null  Int8          
 9   Sexo Vítima         407148 non-null  category      
 10  Cor Vítima          407618 non-null  category      
 11  Data Hora           502414 non-null  datetime64[us]
dtypes: Int8(2), category(8), datetime64[us](1), int64(1)
memory usage: 15.2 MB


### 7. Memory Usage


##### 7.1. After Conversion


In [18]:
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 502414 entries, 0 to 502413
Data columns (total 12 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   Sequência           502414 non-null  int64         
 1   Grupo Fato          502414 non-null  category      
 2   Tipo Enquadramento  502414 non-null  category      
 3   Tipo Fato           502414 non-null  category      
 4   Municipio Fato      502414 non-null  category      
 5   Local Fato          502414 non-null  category      
 6   Bairro              404556 non-null  category      
 7   Quantidade Vítimas  502407 non-null  Int8          
 8   Idade Vítima        407370 non-null  Int8          
 9   Sexo Vítima         407148 non-null  category      
 10  Cor Vítima          407618 non-null  category      
 11  Data Hora           502414 non-null  datetime64[us]
dtypes: Int8(2), category(8), datetime64[us](1), int64(1)
memory usage: 15.2 MB


In [19]:
df["Idade Vítima"].mean()

np.float64(42.39160222893193)

### 8. Loading Data


##### 8.1. Load Function


In [20]:
def load_data(path: str) -> pd.DataFrame:
    """
    Reads a yearly SSP/RS CSV and returns a typed DataFrame. The function reads
    the first 13 columns with latin-1 encoding and ";" separator, merges the date
    and time columns into a single datetime column, drops the original ones, and
    casts the victim count and age columns to nullable Int8.
    """

    df = pd.read_csv(path, sep=";", encoding="latin-1", usecols=range(13))
    df["Data Hora"] = pd.to_datetime(
        df["Data Fato"] + " " + df["Hora Fato"],
        format="%d/%m/%Y %H:%M:%S",
    )
    df = df.drop(columns=["Data Fato", "Hora Fato"])
    integers = ["Quantidade Vítimas", "Idade Vítima"]
    df[integers] = df[integers].astype("Int8")
    return df

In [21]:
df = load_data(path)

In [22]:
df.shape

(502414, 12)

##### 8.2. Function Usage


In [23]:
frames = {}
try:
    for year in range(2021, 2027):
        print(f"Loading {year} data...")
        path = f"../data/{year}_raw.csv"
        frames[year] = load_data(path)
except ValueError as e:
    print(e)


Loading 2021 data...
Loading 2022 data...


/tmp/ipykernel_547901/3123105808.py:9: DtypeWarning: Columns (0: Sequência) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, sep=";", encoding="latin-1", usecols=range(13))


time data "1 30" doesn't match format "%d/%m/%Y %H:%M:%S". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.


##### 8.3. Investigating...

Loading 2022 fails on `"1 30"`: a line break inside the `Bairro` field of record 427926
splits it in two, and the unquoted CSV turns the remainder into a new row shifted left
(victim count in `Data Fato`, age in `Hora Fato`). The fix drops rows whose `Sequência`
isn't numeric, losing only that record's victim fields.


In [24]:
raw = pd.read_csv(
    "../data/2022_raw.csv",
    sep=";",
    encoding="latin-1",
    usecols=range(13),
)
bad = ~raw["Data Fato"].str.match(r"\d{2}/\d{2}/\d{4}")
raw[bad]


/tmp/ipykernel_547901/3135595316.py:1: DtypeWarning: Columns (0: Sequência) have mixed types. Specify dtype option on import or set low_memory=False.
  raw = pd.read_csv(


,Sequência,Data Fato,Hora Fato,Grupo Fato,Tipo Enquadramento,Tipo Fato,Municipio Fato,Local Fato,Bairro,Quantidade Vítimas,Idade Vítima,Sexo Vítima,Cor Vítima
427926,frente ao Postinho da Tieta,1,30,Masculino,Branca,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
raw.iloc[427924:427928]

,Sequência,Data Fato,Hora Fato,Grupo Fato,Tipo Enquadramento,Tipo Fato,Municipio Fato,Local Fato,Bairro,Quantidade Vítimas,Idade Vítima,Sexo Vítima,Cor Vítima
427924,427925,26/07/2022,06:40:00,CRIMES,AMEACA,Consumado,VACARIA,RESIDENCIA,BARCELOS,1.0,57.0,Masculino,Preta
427925,427926,26/07/2022,06:45:00,CRIMES,ROUBO A PEDESTRE,Consumado,TRIUNFO,VIA PUBLICA,NaN,NaN,NaN,NaN,NaN
427926,frente ao Postinho da Tieta,1,30,Masculino,Branca,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
427927,427927,26/07/2022,06:45:00,CRIMES,LESAO CORPORAL CULPOSA DIRECAO VEIC AUTOMOTOR,Consumado,PORTO ALEGRE,VIA PUBLICA,Cristo Redentor,0.0,NaN,NaN,NaN


##### 8.4. Function Fix


In [26]:
def load_data(path: str) -> pd.DataFrame:
    """
    Reads a yearly SSP/RS CSV and returns a typed DataFrame. The function reads
    the first 13 columns with latin-1 encoding and ";" separator, drops malformed
    rows whose "Sequência" is not numeric, merges the date and time columns into a
    single datetime column, drops the original ones, and casts the victim count and
    age columns to nullable Int8.
    """

    df = pd.read_csv(
        path,
        sep=";",
        encoding="latin-1",
        usecols=range(13),
        dtype={"Sequência": str},
    )

    df["Sequência"] = pd.to_numeric(df["Sequência"], errors="coerce")
    df = df[~df["Sequência"].isna()]
    df["Sequência"] = df["Sequência"].astype("int64")

    df["Data Hora"] = pd.to_datetime(
        df["Data Fato"] + " " + df["Hora Fato"],
        format="%d/%m/%Y %H:%M:%S",
    )
    df = df.drop(columns=["Data Fato", "Hora Fato"])

    integers = ["Quantidade Vítimas", "Idade Vítima"]
    df[integers] = df[integers].astype("Int8")

    return df


##### 8.5. Function Usage


In [27]:
frames = {}
for year in range(2021, 2027):
    print(f"Loading {year} data...")
    path = f"../data/{year}_raw.csv"
    frames[year] = load_data(path)

Loading 2021 data...


Loading 2022 data...
Loading 2023 data...
Loading 2024 data...
Loading 2025 data...
Loading 2026 data...
